# Narrador IA — geração de narração com VoxCPM2 (GPU)

Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.

Passos: 1) clonar o projeto, 2) instalar dependências, 3) colar o roteiro, 4) gerar, 5) ouvir/baixar o WAV.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!git clone https://github.com/guzsysdev/narrador.git
%cd narrador

In [ ]:
# O Colab já vem com PyTorch + CUDA instalados — não reinstalamos torch para não
# quebrar essa configuração. Só as dependências específicas do projeto.
!pip install -q voxcpm librosa soundfile

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

## Prévia de voz (opcional, mas recomendado)
Antes de gerar a narração completa (lenta), ouça uma amostra curta de cada voz e escolha qual usar. Gera `previa_masculina.wav` e `previa_feminina.wav`.

In [ ]:
!python cli.py --previa ambas

In [ ]:
from IPython.display import Audio, display
print("Voz masculina:")
display(Audio("previa_masculina.wav"))
print("Voz feminina:")
display(Audio("previa_feminina.wav"))

Gostou de uma delas? Use `--voz` na célula de geração abaixo com a descrição escolhida (ex.: copie a descrição de `sVozPadraoFeminina` em `motorTts.py` se preferir a voz feminina).

## Cole o roteiro completo da narração abaixo
Rode a célula seguinte, cole/edite o texto entre as aspas e execute — isso salva o roteiro em `roteiro.txt`.

In [ ]:
%%writefile roteiro.txt
Cole aqui o roteiro completo da narração, uma linha por ação/passo — cada linha vira um trecho de áudio.

## Gera a narração completa
Ajuste `--velocidade`, `--tom`, `--seed` e `--voz` se quiser. Sem `--cpu`, o script detecta e usa a GPU automaticamente.

In [ ]:
!python cli.py --arquivo-texto roteiro.txt --saida narracao_final.wav --tamanho-trecho 250

In [ ]:
from IPython.display import Audio
Audio("narracao_final.wav")

In [ ]:
from google.colab import files
files.download("narracao_final.wav")

## Alternativa: usar a interface web (em vez do CLI acima)

Roda o `servidorWeb.py` aqui no Colab (com GPU) e expõe pela internet com um túnel (ngrok), pra você usar a interface bonita no navegador em vez de editar o roteiro.txt.

**Pré-requisito:** conta gratuita em https://ngrok.com → pegue seu authtoken em https://dashboard.ngrok.com/get-started/your-authtoken.

Depende das células de clone/checagem de GPU lá em cima já terem rodado.

In [ ]:
!pip install -q fastapi uvicorn pydantic python-multipart pyngrok

In [ ]:
from getpass import getpass
sTokenNgrok = getpass("Cole seu ngrok authtoken (não fica salvo, só nesta sessão): ")

In [ ]:
import subprocess
import time
from pyngrok import ngrok

ngrok.set_auth_token(sTokenNgrok)

# Sobe o servidor em background (o modelo só carrega na primeira geração pedida).
oProcessoServidor = subprocess.Popen(
    ["uvicorn", "servidorWeb:oApp", "--host", "0.0.0.0", "--port", "8000"]
)
time.sleep(3)

oTunel = ngrok.connect(8000)
print("Abra esta URL no navegador:", oTunel.public_url)

Use a interface normalmente (cole o texto, ajuste voz/velocidade/tom, gere e baixe). A URL do ngrok é temporária e muda a cada vez que essa célula é rodada de novo — não compartilhe com terceiros, qualquer um com o link consegue usar o servidor enquanto ele estiver no ar.

### Quando terminar: encerra o servidor e o túnel, pra liberar a GPU/porta

In [ ]:
ngrok.disconnect(oTunel.public_url)
oProcessoServidor.terminate()
print("Servidor e túnel encerrados.")